In [ ]:
api_key=""

In [3]:
!pip install openai pandas tqdm


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# pip install openai tqdm

from openai import OpenAI
from tqdm import tqdm

client = OpenAI(api_key=api_key)

# ==============================
# 1. LOAD WORDS
# ==============================
with open("Spicor_Names.txt", "r", encoding="utf-8") as f:
    words = [w.strip().lower() for w in f if w.strip()]

# ==============================
# 2. CHUNKING
# ==============================
def chunk(lst, size=200):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

# ==============================
# 3. PROMPT
# ==============================
BASE_PROMPT = """
You are a phonetician specializing in Indian English (IndE). Convert the word list into eSpeak-NG
en_extra entries. Use full linguistic judgment — reason from the actual pronunciation. Do not
mechanically apply rules when they produce unnatural output.
== OUTPUT FORMAT ==
word  phoneme_string  // optional comment

One entry per line, word lowercase, sorted alphabetically within this batch.
Single // comment header at top. Inline // only where non-obvious.
Plain UTF-8 only. NEVER use backticks or ``` anywhere. Output must be pasteable as-is.

== PHONEME TABLE — ONLY THESE TOKENS ARE VALID ==
Case is critical. Nothing outside this table exists in the eSpeak English phoneme set.
STRESS:   '  primary    ,  secondary    %  force-unstressed
UTILITY:  |  syllable guard    ||  word break
VOWELS
Token  Sound            IndE use / notes

a      TRAP (short-a)   cat, ankit, bhatt, sakshi — the default short vowel in Indian names
A:     PALM (long-aa)   calm, karma, lata, vandana — long aa sound
A:r    PALM+r           sharma→'SA:rm@  kumar→kUm'A:r  (rhotic long-aa; never A@)
@      schwa            unstressed syllables: rahul→rA:'hUl
@L     syllabic-l       simple, bottle
3:     NURSE            kurta→'k3:tA:  verma→'v3:rmA:
E      DRESS            sen→sEn  hegde→'hEgdE  — bare E, NEVER E:
I      KIT (short-i)    ankit→'aNkIt  vikram→'vIkr@m
i      HAPPY (final-i)  aarti→'A:rti  swati→'swA:ti  prachi→'pratSi — unstressed final
i:     FLEECE (long-ee) deepak→'d|i:p@k  sunita→sUn'i:tA:  rupee→ru:'pi: — stressed long-ee
0      LOT (short-o)    hop
V      STRUT (short-u)  strut
u:     GOOSE (long-oo)  puja→'pu:dZA:  kapoor→k@p'u:r  madhu→m@d'u:
U      FOOT (short-oo)  kumar→kUm'A:r  kunal→kUn'A:l
O:     THOUGHT (aw)     kaur→kO:r  gaur→gO:r
aI     PRICE (eye)      iyer→'aI@r  nayak→n'aIjak  aishwarya→aIS'wA:rj@
eI     FACE (day)       sanjay→'sandZeI  payal→'peIj@l
aU     MOUTH (cow)      gaurav→'gaUr@v  gowda→g'aUdA:
oU     GOAT (go)        rohit→'roUhIt  modi→'moUdi  onkar→'oUNk@r
OI     CHOICE           roy→rOI
i@     NEAR             iyer→'aI@r  (use for -eer/-ear endings)
e@     SQUARE           care
A@     NEAR-variant     valid token but NOT used for IndE rhotic long-aa; use A:r instead
aI@    sequence         aI followed by @; not a single token — write as two phonemes
aU@    sequence         aU followed by @; not a single token — write as two phonemes
U@     sequence         U followed by @; valid eSpeak sequence when needed
CONSONANTS
p  b  t  d  k  g   tS(ch)  dZ(j-sound)   f  v  T(th-thin)  D(th-this)
s  z  S(sh)  Z(zh)   h  m  n  N(ng)   l  r  j(glide)  w  x(loch)
== WHAT IS FORBIDDEN (ALL CAUSE COMPILE ERRORS) ==
NEVER USE   USE INSTEAD  Example

bare A      a / A: / A:r  ankit→'aNkIt  NOT 'ANkIt   bhatt→bat  NOT bA:t
bare 3      3:           only 3: is valid; bare 3 does not exist in English eSpeak
a#  I2  O@  o@           these are not valid eSpeak English phoneme tokens; do not use
H           h            neha→'nEhA:   NOT 'nEH@    sneha→'snEhA:
G R K J Y B C F L M N P W X  (uppercase — use lowercase versions only)
U:          u:           kapoor→k@p'u:r   NOT k@p'U:r
I:          i:           patil→p@t'i:l    NOT p@t'I:l
E:          E            sen→sEn          NOT sE:n
bh dh gh jh kh sh ch ng ph th  (aspirate/digraph sequences — see IndE rules below)
→  b  d  g  dZ k  S  tS N  p  t   (drop aspiration; sh→S; ch→tS; ng→N; ph→p always)
T           t            shetty→'SEti  NOT 'SeTi  (T = th-in-THIN only; never for IndE aspirate-th)
y           j            kavya→k'avj@     NOT k'avy@
a: o:       A: O:        lowercase a: and o: are wrong
æ ə ː       a  @  :      Unicode chars — do not exist in eSpeak English
== IndE RULES ==

Rhotic — always preserve coda r. Use A:r (never A@) for long-aa+r.
Monosyllables — NO stress mark: dal→dA:l  bhatt→bat  nair→neIr  kaur→kO:r  singh→sIN
Multisyllabics — exactly ONE primary stress mark '.
Aspirates — drop aspiration: bh→b  dh→d  gh→g  jh→dZ  kh→k  th→t  sh→S  ch→tS  ph→p
Note: ph→p always in Indian names (never f). T is reserved for genuine English thin-th only.
-ya → j@   -nya → nj@ (plain n, not N)   -jy → dZj   (kavya→k'avj@  ananya→@n'A:nj@  jyoti→'dZjoUti)
iy- onset — "iy" at word start → aI (like "eye"). iyer→'aI@r  iyengar→aI'jENg@r
Word-internal "iy" → j (as in biryani→bIrj'A:ni).
Final unstressed -i/-ee → plain i: aarti→'A:rti  swati→'swA:ti  (NOT i:)
Exception: stressed final -ee keeps i: — rupee→ru:'pi:  naveen→n@v'i:n
Medial stressed -ee- → i: — deepak→'d|i:p@k  sunita→sUn'i:tA:
o → oU whenever stressed or at a syllable onset (never bare o):
onkar→'oUNk@r  rohit→'roUhIt  dosa→d'oUs@
aa- word-initial → A: (long-aa): aarav→'A:r@v  aarti→'A:rti
w in names → w (retained); v in names → v. Do not substitute one for the other.

Word-initial single "a" — three sub-cases:
    (a) Known long-ā Sanskrit stems where first syllable is stressed or heavy → A:
        Only these stems qualify: aditya, anand/ananda, aryan/arya.
        aditya → 'A:dItj@    // ā-ditya; long onset
        anand  → A:'nA:nd    // ā-nanda; long onset
        aryan  → 'A:rj@n     // ā-rya; long onset
    (b) Word-initial ar+consonant that is NOT a long-ā stem → short-a (NOT A:)
        arjun  → 'ardZUn     // short-a; NOT 'A:rdZUn ("Aarjun" is wrong)
        arvind → 'arvInd     // short-a onset
    (c) All other word-initial "a" where stress falls on syllable 2 or later → schwa @
        agarwal → @g@r'wA:l  // stress on -wal; NOT "Aagarwal"
        ajay    → @dZ'eI     // stress on -jay
        amar    → @m'A:r     // stress on -mar
        amit    → @m'It      // stress on -mit
        ashok   → @S'oUk     // stress on -shok
        arun    → @r'Un      // stress on -run
        
== VERIFIED EXAMPLES ==
// Indian English pronunciations — en_extra additions
aditya       'A:dItj@                 // rule 11a — ā-ditya; long onset; -ya→j@
amit         @m'It                    // rule 11c — short onset; stressed i: collapses to I
arav        'A:r@v
arti        'A:rti                   // unstressed final -i; NOT 'A:rti:
agarwal      @g@r'wA:l
aishwarya    aIS'wA:rj@               // aI onset; sh→S
ajay         @dZ'eI
akash        @k'A:S
amar         @m'A:r
ananya       @n'A:nj@                 // -nya→nj@ with plain n
ankit        'aNkIt                   // short-a; N before k; NOT 'ANkIt
arjun        'A:rdZUn
balaji       bA:'ladZi
banerjee     b@n'3:dZ|i:
bhaskar      b'ask@r                  // bh→b; short-a; @r
bhatnagar    bat'nag@r                // bh→b; short-a first syllable; second syllable also short-a
bhatt        bat                      // bh→b; monosyllable; short-a
bharat       b'A:r@t                  // bh→b; long-aa+r
bhosle       b'oUsle
biryani      bIrj'A:ni                // word-internal iy→j
chai         tSaI                     // monosyllable
chatterjee   tS'at@rdZ|i:
contractor   k@ntr'akt@r
crore        kroUr                    // monosyllable
dal          dA:l                     // monosyllable
deepak       'd|i:p@k                 // medial stressed i:
deshmukh     dESm'Uk                  // kh→k; sh→S
dhanush      d'A:nUS                  // dh→d
dosa         d'oUs@                   // o→oU
gaurav       'gaUr@v
ghee         g'i:                      // gh→g; monosyllable
gowda        g'aUdA:
iyer         'aI@r                    // iy-onset→aI; NOT 'aIj@r
iyengar      aI'jENg@r               // iy→aI; j guards boundary
iyangar      aI'jaNg@r
jha          dZA:                     // jh→dZ; monosyllable
joshi        dZ'oUSi
jyoti        'dZjoUti                 // jy→dZj
kapoor       k@p'u:r                  // u: NOT U:
karthik      'kA:rtIk
kaur         kO:r                     // monosyllable; O:+r
kavya        k'avj@                   // -vya→vj@
kulkarni     kUl'kA:rni               // long-aa in -kar-; NOT short-a
kumar        kUm'A:r                  // U=FOOT; A:r=rhotic long-aa
krishnamurti krISn@m'u:rti            // sh→S; -murti has long-u not NURSE vowel
lakshmi      'lakSmi
lata         l'atA:
masala       m@s'A:l@
mishra       'mISrA:
namaste      n@m@st'eI
namrata      n@m'rA:tA:
narayanan    n@r'A:j@n@n
nayak        n'aIjak                  // ay→aI; -yak→jak
nayar        n'aIj@r
neha         'nEhA:                   // h not H
onkar        'oUNk@r                  // o→oU at onset; N before k
pankaj       'paNk@dZ
paneer       p@n'i:r
parthasarathy pA:rt@s@r'A:ti
patil        p@t'i:l                  // i: not I:
payal        'peIj@l                  // eI; NOT paIj@l
pillai       'pIlaI                   // stress on first syllable in common IndE use
prachi       'pratSi
preeti       'pri:ti                  // medial stressed i:; plain i at end
priya        'prIj@
puja         'pu:dZA:
raghav       'rA:g@v                  // gh→g
rahul        rA:'hUl
rajesh       rA:'dZES
rathod       rat'oUd                  // th→t; o→oU
rekha        r'EkA:                   // kh→k
ritu         'ri:tu:
rohit        'roUhIt
rupee        ru:'pi:                  // stressed final i: kept
sachin       's@tSIn
sagar        'sag@r
sakshi       'sakSi
sanjay       'sandZeI
saraswathi   s@r@sw'A:ti             // th→t; w→w; plain final i
saree        'sA:ri:
sarita       s@r'i:tA:
savita       s@v'i:tA:
sen          sEn                      // monosyllable; bare E
shankar      S'aNk@r                  // sh→S; N before k
sharma       'SA:rm@                  // sh→S; A:r rhotic
sheetal      'Si:t@l
shetty       'SEti                    // sh→S; plain t; NEVER T
shreya       'SreIjA:
sidhu        s'Idu:                   // dh→d; final long-u
singh        sIN                      // monosyllable
sneha        'snEhA:
solanki      s'oUlaNki
subramanian  sUbr@m'A:nj@n
sunita       sUn'i:tA:
suresh       sU'rES
swati        'swA:ti
thakkar      t'ak@r                   // th→t; short-a
thakur       tA:k'u:r
tiwari       tIv'A:ri
tripathi     trIp'A:ti                // th→t
trivedi      trIv'Edi
vandana      v@nd'A:nA:
venkatesh    vENk@t'ES
vikram       'vIkr@m
vyas         vjA:s                    // monosyllable; vy→vj
yadav        j'A:d@v
yash         jA:S                     // monosyllable; y→j
== TASK ==
Output ONLY the dictionary entries (single // header + entries).
No prose, no explanation, no code fences, no backticks.
WORDS:
[INSERT YOUR WORDS HERE, ONE PER LINE]
"""

def build_prompt(batch):
    return BASE_PROMPT + "\n\nWORDS:\n" + "\n".join(batch)

# ==============================
# 4. API CALL
# ==============================
# def generate(batch):
#     response = client.chat.completions.create(
#         model="gpt-4o",
#         temperature=0,
#         messages=[{"role": "user", "content": build_prompt(batch)}]
#     )
#     return response.choices[0].message.content
def generate(batch):
    response = client.responses.create(
        model="gpt-5.4",
        temperature=0,
        top_p=1,
        input=[
            {
                "role": "system",
                "content": (
                    "You are a phonetician specializing in Indian English. "
                    "STRICTLY follow phoneme rules. "
                    "If ANY output violates rules, fix it before returning."
                )
            },
            {
                "role": "user",
                "content": build_prompt(batch)
            }
        ]
    )
    return response.output[0].content[0].text
# ==============================
# 5. PROCESS
# ==============================
all_outputs = []

for batch in tqdm(list(chunk(words, 200))):
    try:
        output = generate(batch)
        all_outputs.append(output.strip())
    except Exception as e:
        print(f"Error: {e}")

# ==============================
# 6. SAVE FINAL FILE
# ==============================
with open("espeak_phoneme_Spicor_Names.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(all_outputs))

print("Done. Output saved to espeak_dict_corrected.txt")

100%|██████████████████████████████████████████████████████████████████████████████████| 40/40 [23:36<00:00, 35.40s/it]

Done. Output saved to espeak_dict_corrected.txt
